<a href="https://colab.research.google.com/github/SilviaPosso78/CreacionDataset/blob/main/Carga_y_Creaci%C3%B3n_de_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad 1: Obtener y preparar datos desde Hugging Face

Este notebook coordina el proceso de descarga de un dataset desde Hugging Face, la aplicación de transformaciones de datos utilizando pandas, y la exportación de un resumen estadístico y estructural a un archivo `.txt`.

In [ ]:
# Instalación de dependencias necesarias
!pip install datasets pandas

## 1. Creación de las Clases en Python

Creamos el archivo `hf_loader.py` que contiene la clase encargada de conectarse a Hugging Face y extraer el dataset como un DataFrame de Pandas.

In [23]:
%%writefile hf_loader.py
import pandas as pd
from datasets import load_dataset

class HFDatasetLoader:
    def __init__(self, dataset_name: str, split: str = "train"):
        self.dataset_name = dataset_name
        self.split = split

    def load_to_dataframe(self) -> pd.DataFrame:
        """Carga el dataset desde Hugging Face y lo convierte en DataFrame de pandas."""
        dataset = load_dataset(self.dataset_name, split=self.split)
        return dataset.to_pandas()

Overwriting hf_loader.py


Creamos el archivo `summarizer.py` que contiene la clase encargada de analizar un DataFrame y generar un reporte de resumen en formato `.txt`.

In [6]:
%%writefile dataset_summarizer.py
import pandas as pd
import io

class DatasetSummarizer:
    def __init__(self, df: pd.DataFrame):
        self.df = df

    def generate_summary(self, output_filename: str = "resumen_dataset.txt"):
        """Genera un archivo .txt con la estructura, tipos de datos, nulos y estadísticas."""
        buffer = io.StringIO()
        self.df.info(buf=buffer)
        info_str = buffer.getvalue()

        summary_content = []
        summary_content.append("=" * 50)
        summary_content.append("RESUMEN DEL DATAFRAME TRANSFORMADO")
        summary_content.append("=" * 50)
        summary_content.append(f"\nDimensiones: {self.df.shape[0]} filas, {self.df.shape[1]} columnas\n")

        summary_content.append("--- INFORMACIÓN GENERAL Y TIPOS DE DATOS ---")
        summary_content.append(info_str)

        summary_content.append("\n--- VALORES NULOS POR COLUMNA ---")
        summary_content.append(str(self.df.isnull().sum()))

        summary_content.append("\n--- ESTADÍSTICAS DESCRIPTIVAS ---")
        summary_content.append(str(self.df.describe(include='all')))

        with open(output_filename, "w", encoding="utf-8") as f:
            f.write("\n".join(summary_content))

        print(f"Resumen guardado exitosamente en '{output_filename}'")

Overwriting summarizer.py


## 2. Ejecución, Carga y Orquestación del Flujo

In [26]:
# Importamos las clases creadas desde nuestros módulos locales .py
from hf_loader import HFDatasetLoader
from summarizer import DatasetSummarizer

# Cargamos el dataset utilizando la clase HFDatasetLoader
# Usaremos 'scikit-learn/iris' como ejemplo
loader = HFDatasetLoader(dataset_name="scikit-learn/iris", split="train")
df_original = loader.load_to_dataframe()

# Mostramos las primeras filas originales
display(df_original.head())

Cargando el dataset 'scikit-learn/iris' (split: 'train') desde Hugging Face...
¡Dataset cargado exitosamente!


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## 3. Aplicación de 4 Transformaciones Distintas

In [11]:
# Copiamos el dataframe para no modificar el original directamente
df_transformed = df_original.copy()

# --- Transformación 1: Renombrar columnas a español para mayor claridad ---
# Usamos los nombres reales de las columnas del dataset 'scikit-learn/iris' cargado
column_mapping = {
    'SepalLengthCm': 'largo_sepalo_cm',
    'SepalWidthCm': 'ancho_sepalo_cm',
    'PetalLengthCm': 'largo_petalo_cm',
    'PetalWidthCm': 'ancho_petalo_cm',
    'Species': 'especie'
}
df_transformed = df_transformed.rename(columns=column_mapping)
print("1. Columnas renombradas:")
display(df_transformed.head(2))

# --- Transformación 2: Filtrar filas basadas en una condición ---
# Filtramos para quedarnos únicamente con registros que tengan un largo de sépalo mayor a 5.0 cm
df_transformed = df_transformed[df_transformed['largo_sepalo_cm'] > 5.0]
print(f"2. DataFrame filtrado (largo_sepalo_cm > 5.0). Nuevas dimensiones: {df_transformed.shape}")

# --- Transformación 3: Mapear valores de la especie a nombres más limpios ---
# Las especies originales vienen como 'Iris-setosa', 'Iris-versicolor', 'Iris-virginica'.
# Las mapeamos a nombres más simples/cortos.
species_map = {
    'Iris-setosa': 'setosa',
    'Iris-versicolor': 'versicolor',
    'Iris-virginica': 'virginica'
}
df_transformed['especie'] = df_transformed['especie'].map(species_map)
print("3. Mapeo de especies aplicado:")
display(df_transformed.head(2))

# --- Transformación 4: Crear una nueva columna calculada ---
# Calculamos la proporción/ratio entre el largo y ancho del pétalo
df_transformed['ratio_petalo'] = df_transformed['largo_petalo_cm'] / df_transformed['ancho_petalo_cm']
print("4. Columna calculada 'ratio_petalo' agregada:")
display(df_transformed.head(5))

1. Columnas renombradas:


,Id,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa


2. DataFrame filtrado (largo_sepalo_cm > 5.0). Nuevas dimensiones: (118, 6)
3. Mapeo de especies aplicado:


,Id,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie
0,1,5.1,3.5,1.4,0.2,setosa
5,6,5.4,3.9,1.7,0.4,setosa


4. Columna calculada 'ratio_petalo' agregada:


,Id,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie,ratio_petalo
0,1,5.1,3.5,1.4,0.2,setosa,7.00
5,6,5.4,3.9,1.7,0.4,setosa,4.25
10,11,5.4,3.7,1.5,0.2,setosa,7.50
14,15,5.8,4.0,1.2,0.2,setosa,6.00
15,16,5.7,4.4,1.5,0.4,setosa,3.75


## 4. Resumen y Exportación

Finalmente, enviamos el DataFrame transformado a la clase `DatasetSummarizer` para generar el reporte de texto final.

In [27]:
# Instanciar el resumidor con el dataframe resultante
summarizer = DatasetSummarizer(df_transformed)

# Generar el archivo txt de salida
summarizer.generate_txt_summary("resumen_dataset.txt")

Resumen exportado exitosamente a: 'resumen_dataset.txt'


Mostremos el contenido del archivo generado como comprobación final del correcto funcionamiento:

In [ ]:
with open("resumen_dataset.txt", "r") as file:
    print(file.read())

# Actividad 2: Manejo de archivos CSV, JSON y TXT

Este apartado implementa la lectura, modificación y escritura de archivos locales en formatos CSV y JSON mediante programación orientada a objetos (POO) en Python, y consolida las métricas de procesamiento en un reporte final `.txt`.

## 1. Descarga/Creación de los Archivos de Datos de Prueba (Simulando Hugging Face)

Crearemos localmente un archivo `datos_originales.csv` y un `datos_originales.json` emulando datasets estructurados de Hugging Face.

In [14]:
%%writefile datos_originales.csv
id,nombre,edad,ciudad
1,Ana,25,Bogota
2,Luis,30,Medellin
3,Carlos,22,Cali

Writing datos_originales.csv


In [15]:
%%writefile datos_originales.json
[
  {"id": 1, "producto": "Laptop", "precio": 1200, "stock": 10},
  {"id": 2, "producto": "Celular", "precio": 800, "stock": 25},
  {"id": 3, "producto": "Audifonos", "precio": 150, "stock": 50}
]

Writing datos_originales.json


## 2. Definición de Clases en archivos `.py`

In [16]:
%%writefile manejador_csv.py
import csv

class ManejadorCSV:
    def __init__(self, ruta_archivo: str):
        self.ruta_archivo = ruta_archivo
        self.datos = []
        self.encabezados = []

    def importar(self):
        with open(self.ruta_archivo, mode='r', encoding='utf-8') as f:
            lector = csv.reader(f)
            self.encabezados = next(lector)
            self.datos = list(lector)
        print(f"CSV importado correctamente: {len(self.datos)} registros cargados.")

    def modificar_registro(self, id_registro: str, columna: str, nuevo_valor: str):
        # Buscar el índice de la columna solicitada
        idx_col = self.encabezados.index(columna)
        modificado = False
        for fila in self.datos:
            if fila[0] == id_registro:  # Asumiendo 'id' en la primera columna
                fila[idx_col] = nuevo_valor
                modificado = True
                break
        if modificado:
            print(f"Registro ID {id_registro} modificado: {columna} -> {nuevo_valor}.")
        else:
            print(f"No se encontró el registro con ID {id_registro}.")

    def guardar(self, nueva_ruta: str):
        with open(nueva_ruta, mode='w', newline='', encoding='utf-8') as f:
            escritor = csv.writer(f)
            escritor.writerow(self.encabezados)
            escritor.writerows(self.datos)
        print(f"Nuevo archivo CSV guardado exitosamente en: {nueva_ruta}")

Writing manejador_csv.py


In [17]:
%%writefile manejador_json.py
import json

class ManejadorJSON:
    def __init__(self, ruta_archivo: str):
        self.ruta_archivo = ruta_archivo
        self.datos = []

    def importar(self):
        with open(self.ruta_archivo, mode='r', encoding='utf-8') as f:
            self.datos = json.load(f)
        print(f"JSON importado correctamente: {len(self.datos)} objetos cargados.")

    def agregar_registro(self, nuevo_registro: dict):
        self.datos.append(nuevo_registro)
        print(f"Nuevo registro agregado al JSON: {nuevo_registro['producto'] if 'producto' in nuevo_registro else ''}")

    def guardar(self, nueva_ruta: str):
        with open(nueva_ruta, mode='w', encoding='utf-8') as f:
            json.dump(self.datos, f, indent=2, ensure_ascii=False)
        print(f"Nuevo archivo JSON guardado exitosamente en: {nueva_ruta}")

Writing manejador_json.py


In [18]:
%%writefile generador_resumen_txt.py
class GeneradorResumenTXT:
    def __init__(self, url_csv: str, url_json: str, cambios_csv: str, cambios_json: str, registros_csv: int, registros_json: int, ruta_salida_csv: str, ruta_salida_json: str):
        self.url_csv = url_csv
        self.url_json = url_json
        self.cambios_csv = cambios_csv
        self.cambios_json = cambios_json
        self.registros_csv = registros_csv
        self.registros_json = registros_json
        self.ruta_salida_csv = ruta_salida_csv
        self.ruta_salida_json = ruta_salida_json

    def generar_reporte(self, ruta_resumen: str = "resumen_ae2.txt"):
        with open(ruta_resumen, "w", encoding="utf-8") as f:
            f.write("==================================================\n")
            f.write("          REPORTE DE PROCESAMIENTO AE2            \n")
            f.write("==================================================\n\n")
            f.write(f"1. FUENTES DE DATOS ORIGEN (Hugging Face):\n")
            f.write(f"   - URL Origen CSV: {self.url_csv}\n")
            f.write(f"   - URL Origen JSON: {self.url_json}\n\n")
            f.write(f"2. METADATOS DE PROCESAMIENTO:\n")
            f.write(f"   - Cantidad registros CSV procesados: {self.registros_csv}\n")
            f.write(f"   - Cantidad registros JSON procesados: {self.registros_json}\n\n")
            f.write(f"3. MODIFICACIONES IMPLEMENTADAS:\n")
            f.write(f"   - Modificación en CSV: {self.cambios_csv}\n")
            f.write(f"   - Modificación en JSON: {self.cambios_json}\n\n")
            f.write(f"4. ARCHIVOS GENERADOS Y GUARDADOS:\n")
            f.write(f"   - Ruta de destino CSV: {self.ruta_salida_csv}\n")
            f.write(f"   - Ruta de destino JSON: {self.ruta_salida_json}\n")
        print(f"Reporte de auditoría generado correctamente en: '{ruta_resumen}'")

Writing generador_resumen_txt.py


## 3. Orquestación y Ejecución de Flujo

In [19]:
# Definición de URLs de Origen
url_csv_origen = "https://huggingface.co/datasets/sample/raw/main/datos_originales.csv"
url_json_origen = "https://huggingface.co/datasets/sample/raw/main/datos_originales.json"

# Importación de Clases
from manejador_csv import ManejadorCSV
from manejador_json import ManejadorJSON
from generador_resumen_txt import GeneradorResumenTXT

# --- Procesar CSV ---
print("=== PROCESANDO CSV ===")
csv_handler = ManejadorCSV("datos_originales.csv")
csv_handler.importar()
print("Datos originales del CSV:", csv_handler.datos)

# Aplicar Modificación
csv_handler.modificar_registro(id_registro="2", columna="ciudad", nuevo_valor="Barranquilla")
print("Datos modificados del CSV:", csv_handler.datos)

# Guardar
csv_handler.guardar("datos_modificados.csv")

# --- Procesar JSON ---
print("\n=== PROCESANDO JSON ===")
json_handler = ManejadorJSON("datos_originales.json")
json_handler.importar()
print("Datos originales del JSON:", json_handler.datos)

# Aplicar Modificación (Añadir un elemento)
nuevo_item = {"id": 4, "producto": "Teclado Mecanico", "precio": 120, "stock": 15}
json_handler.agregar_registro(nuevo_item)
print("Datos modificados del JSON:", json_handler.datos)

# Guardar
json_handler.guardar("datos_modificados.json")

=== PROCESANDO CSV ===
CSV importado correctamente: 3 registros cargados.
Datos originales del CSV: [['1', 'Ana', '25', 'Bogota'], ['2', 'Luis', '30', 'Medellin'], ['3', 'Carlos', '22', 'Cali']]
Registro ID 2 modificado: ciudad -> Barranquilla.
Datos modificados del CSV: [['1', 'Ana', '25', 'Bogota'], ['2', 'Luis', '30', 'Barranquilla'], ['3', 'Carlos', '22', 'Cali']]
Nuevo archivo CSV guardado exitosamente en: datos_modificados.csv

=== PROCESANDO JSON ===
JSON importado correctamente: 3 objetos cargados.
Datos originales del JSON: [{'id': 1, 'producto': 'Laptop', 'precio': 1200, 'stock': 10}, {'id': 2, 'producto': 'Celular', 'precio': 800, 'stock': 25}, {'id': 3, 'producto': 'Audifonos', 'precio': 150, 'stock': 50}]
Nuevo registro agregado al JSON: Teclado Mecanico
Datos modificados del JSON: [{'id': 1, 'producto': 'Laptop', 'precio': 1200, 'stock': 10}, {'id': 2, 'producto': 'Celular', 'precio': 800, 'stock': 25}, {'id': 3, 'producto': 'Audifonos', 'precio': 150, 'stock': 50}, {'id'

## 4. Generación y Visualización del Resumen Final

In [20]:
# Generación del Resumen
resumidor = GeneradorResumenTXT(
    url_csv=url_csv_origen,
    url_json=url_json_origen,
    cambios_csv="Se actualizó la ciudad del ID 2 de 'Medellin' a 'Barranquilla'",
    cambios_json="Se agregó un nuevo producto de ID 4 (Teclado Mecanico)",
    registros_csv=len(csv_handler.datos),
    registros_json=len(json_handler.datos),
    ruta_salida_csv="datos_modificados.csv",
    ruta_salida_json="datos_modificados.json"
)

resumidor.generar_reporte("resumen_ae2.txt")

# Mostrar reporte generado
print("\n--- CONTENIDO DEL ARCHIVO GENERADO ---")
with open("resumen_ae2.txt", "r", encoding="utf-8") as f:
    print(f.read())

Reporte de auditoría generado correctamente en: 'resumen_ae2.txt'

--- CONTENIDO DEL ARCHIVO GENERADO ---
          REPORTE DE PROCESAMIENTO AE2            

1. FUENTES DE DATOS ORIGEN (Hugging Face):
   - URL Origen CSV: https://huggingface.co/datasets/sample/raw/main/datos_originales.csv
   - URL Origen JSON: https://huggingface.co/datasets/sample/raw/main/datos_originales.json

2. METADATOS DE PROCESAMIENTO:
   - Cantidad registros CSV procesados: 3
   - Cantidad registros JSON procesados: 4

3. MODIFICACIONES IMPLEMENTADAS:
   - Modificación en CSV: Se actualizó la ciudad del ID 2 de 'Medellin' a 'Barranquilla'
   - Modificación en JSON: Se agregó un nuevo producto de ID 4 (Teclado Mecanico)

4. ARCHIVOS GENERADOS Y GUARDADOS:
   - Ruta de destino CSV: datos_modificados.csv
   - Ruta de destino JSON: datos_modificados.json

